# Vehicle Detection with HOG + SVM

Dataset: YOLOv9 format (grayscale), 5 classes — Bus, Car, Motorcycle, Pickup, Truck

Pipeline: EDA, outlier filtering, positive/negative extraction, HOG features, PCA, SVM, Hard Negative Mining, sliding-window detection

## 1. Imports & Configuration

In [ ]:
import cv2
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, precision_recall_curve)
from imblearn.over_sampling import SMOTE

DATASET_PATH = Path('dataset/Apply_Grayscale/Apply_Grayscale/Vehicles_Detection.v9i.yolov9')
MODEL_DIR    = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ['Bus', 'Car', 'Motorcycle', 'Pickup', 'Truck']
COLORS      = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]
W = H       = 640
WINDOW_SIZE = (64, 64)

print(f'Dataset : {DATASET_PATH}')
print(f'Classes : {CLASS_NAMES}')

## 2. Load Labels

In [ ]:
def load_labels(img_dir, lbl_dir):
    rows = []
    for lbl_file in sorted(lbl_dir.glob('*.txt')):
        img_file = img_dir / (lbl_file.stem + '.jpg')
        if not img_file.exists():
            continue
        lines = lbl_file.read_text().splitlines()
        if not lines:
            rows.append({'img_path': str(img_file), 'class_id': -1,
                         'class_name': 'None', 'cx': -1, 'cy': -1,
                         'w': -1, 'h': -1, 'has_obj': False})
        else:
            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    continue
                cid = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                rows.append({'img_path': str(img_file), 'class_id': cid,
                             'class_name': CLASS_NAMES[cid],
                             'cx': cx, 'cy': cy, 'w': bw, 'h': bh,
                             'has_obj': True})
    return pd.DataFrame(rows)


splits = {}
for s in ['train', 'valid', 'test']:
    splits[s] = load_labels(DATASET_PATH / s / 'images', DATASET_PATH / s / 'labels')
    print(f'{s:>5}: {splits[s]["img_path"].nunique():>3} images, {splits[s]["has_obj"].sum():>4} boxes')

df_all = pd.concat(splits, names=['split']).reset_index(level=0)
print(f'Total : {df_all["img_path"].nunique()} images, {df_all["has_obj"].sum()} boxes')

## 3. EDA: Class Distribution

In [ ]:
rgb_colors = [(c[2]/255, c[1]/255, c[0]/255) for c in COLORS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (sname, df) in enumerate(splits.items()):
    counts = (df[df['has_obj']]['class_name']
              .value_counts().reindex(CLASS_NAMES, fill_value=0))
    axes[0].bar([i + idx * 0.25 for i in range(len(CLASS_NAMES))],
                counts.values, width=0.2, label=sname, alpha=0.8)
axes[0].set_xticks(range(len(CLASS_NAMES)))
axes[0].set_xticklabels(CLASS_NAMES)
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution per Split')
axes[0].legend()

total_obj    = df_all[df_all['has_obj']]
total_counts = total_obj.groupby('class_name').size().reindex(CLASS_NAMES, fill_value=0)
bars = axes[1].bar(CLASS_NAMES, total_counts.values, color=rgb_colors)
axes[1].set_ylabel('Count')
axes[1].set_title('Total Class Distribution')
for bar, v in zip(bars, total_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 str(v), ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f"{'Class':<15} {'Train':>6} {'Valid':>6} {'Test':>6} {'Total':>6} {'%':>7}")
print('-' * 55)
for c in CLASS_NAMES:
    tr = (splits['train'][splits['train']['has_obj']]['class_name'] == c).sum()
    va = (splits['valid'][splits['valid']['has_obj']]['class_name'] == c).sum()
    te = (splits['test'] [splits['test'] ['has_obj']]['class_name'] == c).sum()
    tot = tr + va + te
    print(f'{c:<15} {tr:>6} {va:>6} {te:>6} {tot:>6} {100*tot/len(total_obj):>6.2f}%')

## 4. EDA: Boxes per Image

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for sname, df in splits.items():
    bpi = df[df['has_obj']].groupby('img_path').size()
    ax.hist(bpi, bins=20, alpha=0.5, label=sname)
ax.set_xlabel('Bounding boxes per image')
ax.set_ylabel('Frequency')
ax.set_title('Box Count Distribution')
ax.legend()
plt.show()

print(f"{'Split':<10} {'Min':>5} {'Max':>5} {'Mean':>6} {'Median':>7}")
print('-' * 38)
for sname, df in splits.items():
    bpi = df[df['has_obj']].groupby('img_path').size()
    print(f'{sname:<10} {bpi.min():>5} {bpi.max():>5} {bpi.mean():>6.2f} {bpi.median():>7.1f}')

## 5. EDA: Box Size & Aspect Ratio

In [ ]:
rows = []
for sname, df in splits.items():
    for img_path, group in df[df['has_obj']].groupby('img_path'):
        for _, row in group.iterrows():
            w_px = row['w'] * W
            h_px = row['h'] * H
            rows.append({
                'split'     : sname,
                'class_name': row['class_name'],
                'class_id'  : row['class_id'],
                'w_px'  : w_px,
                'h_px'  : h_px,
                'area'  : w_px * h_px,
                'aspect': w_px / h_px if h_px > 0 else 0,
            })
df_size = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for cid, cname in enumerate(CLASS_NAMES):
    sub = df_size[df_size['class_name'] == cname]
    axes[0].scatter(sub['w_px'], sub['h_px'], c=[rgb_colors[cid]],
                    label=cname, alpha=0.4, s=8)
axes[0].set_xlabel('Width (px)'); axes[0].set_ylabel('Height (px)')
axes[0].set_title('Box Dimensions by Class'); axes[0].legend(markerscale=3)

axes[1].hist(np.log10(df_size['area'] + 1), bins=40, edgecolor='black')
axes[1].set_xlabel('log10(Area + 1)'); axes[1].set_title('Box Area Distribution')

axes[2].hist(np.clip(df_size['aspect'], 0, 5), bins=40, edgecolor='black')
axes[2].set_xlabel('Aspect Ratio (w/h)'); axes[2].set_title('Aspect Ratio Distribution')

plt.tight_layout(); plt.show()

print(f"{'Metric':<20} {'Min':>8} {'25%':>8} {'50%':>8} {'75%':>8} {'Max':>8}")
print('-' * 60)
for m in ['w_px', 'h_px', 'area', 'aspect']:
    q = df_size[m].quantile([0, .25, .5, .75, 1]).values
    print(f'{m:<20} {q[0]:>8.1f} {q[1]:>8.1f} {q[2]:>8.1f} {q[3]:>8.1f} {q[4]:>8.1f}')

## 6. Outlier Filtering

In [ ]:
MIN_AREA   = max(df_size['area'].quantile(0.01),    200)
MIN_ASPECT = max(df_size['aspect'].quantile(0.01),   0.3)
MAX_ASPECT = min(df_size['aspect'].quantile(0.99),   4.0)

print(f'MIN_AREA   = {MIN_AREA:.0f} px2')
print(f'MIN_ASPECT = {MIN_ASPECT:.2f}')
print(f'MAX_ASPECT = {MAX_ASPECT:.2f}')

df_filtered = df_size[
    (df_size['area']   >= MIN_AREA) &
    (df_size['aspect'] >= MIN_ASPECT) &
    (df_size['aspect'] <= MAX_ASPECT)
].copy()

before, after = len(df_size), len(df_filtered)
print(f'Before: {before} boxes  After: {after} boxes ({before-after} removed, {(before-after)/before*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for label, d in [('Before', df_size), ('After', df_filtered)]:
    axes[0].hist(np.log10(d['area'] + 1), bins=40, alpha=0.5, label=label)
    axes[1].hist(np.clip(d['aspect'], 0, 5), bins=40, alpha=0.5, label=label)
axes[0].set_xlabel('log10(Area)');  axes[0].set_title('Area: Before vs After');  axes[0].legend()
axes[1].set_xlabel('Aspect Ratio'); axes[1].set_title('Aspect: Before vs After'); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Sample Annotation Visualisation

In [ ]:
def draw_boxes(img_path, df_row, colors):
    img = cv2.imread(img_path)
    if img is None:
        return None
    for _, row in df_row.iterrows():
        if not row['has_obj']:
            continue
        x1 = int((row['cx'] - row['w'] / 2) * W)
        y1 = int((row['cy'] - row['h'] / 2) * H)
        x2 = int((row['cx'] + row['w'] / 2) * W)
        y2 = int((row['cy'] + row['h'] / 2) * H)
        color = colors[row['class_id']]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = row['class_name']
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x1, y1 - th - 4), (x1 + tw + 4, y1), color, -1)
        cv2.putText(img, label, (x1 + 2, y1 - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for row_idx, sname in enumerate(['train', 'valid', 'test']):
    df = splits[sname]
    samples = random.sample(list(df['img_path'].unique()), min(3, df['img_path'].nunique()))
    for col_idx, ipath in enumerate(samples):
        img_df = df[df['img_path'] == ipath]
        n_boxes = int(img_df['has_obj'].sum())
        axes[row_idx][col_idx].imshow(draw_boxes(ipath, img_df, COLORS))
        axes[row_idx][col_idx].set_title(f'{sname}: {Path(ipath).name} ({n_boxes} obj)')
        axes[row_idx][col_idx].axis('off')
plt.tight_layout()
plt.show()

## 8. Positive & Negative Sample Extraction

In [ ]:
def preprocess(crop, target_size=WINDOW_SIZE):
    gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.resize(clahe.apply(gray), target_size, interpolation=cv2.INTER_AREA)


def random_crop_bg(img_path, boxes_px, target_size=WINDOW_SIZE, max_tries=50):
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    ws, hs = target_size
    for _ in range(max_tries):
        x = random.randint(0, max(1, w - ws))
        y = random.randint(0, max(1, h - hs))
        ok = True
        for bx1, by1, bx2, by2 in boxes_px:
            ix1, iy1 = max(x, bx1), max(y, by1)
            ix2, iy2 = min(x + ws, bx2), min(y + hs, by2)
            if ix1 < ix2 and iy1 < iy2:
                inter = (ix2 - ix1) * (iy2 - iy1)
                union = ws * hs + (bx2 - bx1) * (by2 - by1) - inter
                if inter / union > 0.2:
                    ok = False
                    break
        if ok:
            return preprocess(img[y:y + hs, x:x + ws])
    return None


df_train    = splits['train']
pos_samples = []
neg_samples = []

for img_path in df_train['img_path'].unique():
    img_df = df_train[df_train['img_path'] == img_path]
    img    = cv2.imread(img_path)
    if img is None:
        continue
    boxes_px = []
    for _, row in img_df.iterrows():
        if not row['has_obj']:
            continue
        x1 = int((row['cx'] - row['w'] / 2) * W)
        y1 = int((row['cy'] - row['h'] / 2) * H)
        x2 = int((row['cx'] + row['w'] / 2) * W)
        y2 = int((row['cy'] + row['h'] / 2) * H)
        if (x2-x1)*(y2-y1) < MIN_AREA:
            continue
        asp = (x2-x1)/(y2-y1) if (y2-y1) > 0 else 0
        if not (MIN_ASPECT <= asp <= MAX_ASPECT):
            continue
        boxes_px.append((x1, y1, x2, y2))
        crop = preprocess(img[y1:y2, x1:x2])
        pos_samples.append(crop)
        if random.random() < 0.5:
            pos_samples.append(cv2.flip(crop, 1))
        # extra augmentation for minority classes
        if row['class_name'] in ('Bus', 'Truck', 'Pickup') and random.random() < 0.5:
            alpha = random.uniform(0.7, 1.3)
            pos_samples.append(np.clip(crop.astype(np.float32) * alpha, 0, 255).astype(np.uint8))

    for _ in range(len(boxes_px) * 5):
        neg = random_crop_bg(img_path, boxes_px)
        if neg is not None:
            neg_samples.append(neg)

print(f'Positive samples: {len(pos_samples)}')
print(f'Negative samples: {len(neg_samples)}')

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(6):
    if i < len(pos_samples):
        axes[0][i].imshow(pos_samples[i], cmap='gray'); axes[0][i].set_title(f'Pos {i+1}')
    if i < len(neg_samples):
        axes[1][i].imshow(neg_samples[i], cmap='gray'); axes[1][i].set_title(f'Neg {i+1}')
    axes[0][i].axis('off'); axes[1][i].axis('off')
plt.suptitle('Positive (top) vs Negative (bottom) Samples')
plt.tight_layout(); plt.show()

## 9. HOG Feature Extraction + PCA

In [ ]:
HOG_WIN    = WINDOW_SIZE
HOG_BLOCK  = (16, 16)
HOG_STRIDE = (8, 8)
HOG_CELL   = (8, 8)
HOG_BINS   = 9

def create_hog():
    return cv2.HOGDescriptor(HOG_WIN, HOG_BLOCK, HOG_STRIDE, HOG_CELL, HOG_BINS)

hog = create_hog()
print(f'Feature size per sample: {hog.getDescriptorSize()}')

X_pos = np.array([hog.compute(s).flatten() for s in pos_samples])
X_neg = np.array([hog.compute(s).flatten() for s in neg_samples])
X = np.vstack([X_pos, X_neg])
y = np.hstack([np.ones(len(X_pos)), np.zeros(len(X_neg))])
print(f'X: {X.shape}  Pos: {int(y.sum())}  Neg: {int((1-y).sum())}')

# Reduce 1764 -> 200 dims
pca   = PCA(n_components=200, random_state=42)
X_pca = pca.fit_transform(X)
print(f'After PCA: {X_pca.shape}')
pickle.dump(pca, open(MODEL_DIR / 'pca.pkl', 'wb'))

X_train, X_val, y_train, y_val = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}')

## 10. SVM Training (SMOTE + GridSearchCV)

In [ ]:
print('Training SVM: SMOTE + GridSearchCV (RBF)...')

smote        = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print(f'After SMOTE  Pos: {int(y_res.sum())}  Neg: {int((1-y_res).sum())}')

scaler       = StandardScaler()
X_res_scaled = scaler.fit_transform(X_res)
X_val_scaled = scaler.transform(X_val)

param_grid = {'C': [0.01, 0.1, 1], 'gamma': ['scale', 'auto']}
cv         = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
gs         = GridSearchCV(
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    param_grid, cv=cv, scoring='f1', n_jobs=-1
)
gs.fit(X_res_scaled, y_res)
print(f'Best params : {gs.best_params_}')
print(f'Best CV F1  : {gs.best_score_:.4f}')

clf = gs.best_estimator_
pickle.dump(clf,    open(MODEL_DIR / 'svm_hog.pkl', 'wb'))
pickle.dump(scaler, open(MODEL_DIR / 'scaler.pkl',  'wb'))
print('Models saved to models/')

## 11. Model Evaluation

In [ ]:
y_pred = clf.predict(X_val_scaled)

print('Classification Report:')
print(classification_report(y_val, y_pred, target_names=['Background', 'Vehicle']))

cm   = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Background', 'Vehicle'])
disp.plot()
plt.title('Confusion Matrix - Validation Set')
plt.show()

tp, fp = cm[1, 1], cm[0, 1]
fn, tn = cm[1, 0], cm[0, 0]
acc  = (tp + tn) / cm.sum()
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
print(f'Accuracy : {acc:.3f}')
print(f'Precision: {prec:.3f}')
print(f'Recall   : {rec:.3f}')
print(f'F1-score : {f1:.3f}')

In [ ]:
scores_val = clf.decision_function(X_val_scaled)
pos_scores = scores_val[y_val == 1]
neg_scores = scores_val[y_val == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(pos_scores, bins=40, alpha=0.6, color='green', label='Vehicle')
axes[0].hist(neg_scores, bins=40, alpha=0.6, color='red',   label='Background')
axes[0].axvline(0, color='black', linestyle='--', label='Default boundary')
axes[0].set_xlabel('Decision function score')
axes[0].set_title('Score Distribution by Class')
axes[0].legend()

precisions, recalls, _ = precision_recall_curve(y_val, scores_val)
axes[1].plot(recalls, precisions, 'b-')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].grid(True)
plt.tight_layout(); plt.show()

# Find threshold that maximises F1
best_f1, best_th = 0, 0
for th in np.linspace(scores_val.min(), scores_val.max(), 200):
    yp  = (scores_val >= th).astype(int)
    tp_ = ((y_val == 1) & (yp == 1)).sum()
    fp_ = ((y_val == 0) & (yp == 1)).sum()
    fn_ = ((y_val == 1) & (yp == 0)).sum()
    p_  = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0
    r_  = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0
    f1_ = 2 * p_ * r_ / (p_ + r_) if (p_ + r_) > 0 else 0
    if f1_ > best_f1:
        best_f1, best_th = f1_, th

print(f'Optimal threshold : {best_th:.3f}')
print(f'F1 at threshold   : {best_f1:.3f}')
print(f'Recall at threshold: {((scores_val >= best_th) & (y_val == 1)).sum() / (y_val == 1).sum():.3f}')
print(f'\nUse conf_thresh={best_th:.3f} in the detect() function below.')

## 12. Hard Negative Mining

In [ ]:
def iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter  = max(0, xB - xA) * max(0, yB - yA)
    areaA  = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB  = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union  = areaA + areaB - inter
    return inter / union if union > 0 else 0


def nms(detections, iou_thresh=0.4):
    dets, keep = sorted(detections, key=lambda x: x[4], reverse=True), []
    while dets:
        best = dets.pop(0)
        keep.append(best)
        dets = [d for d in dets if iou(best[:4], d[:4]) <= iou_thresh]
    return keep


def sliding_window(img, clf, hog, scaler, pca, win_size=WINDOW_SIZE, stride=32, scale=1.0):
    detections = []
    h, w = img.shape[:2]
    if h < win_size[1] or w < win_size[0]:
        return detections
    if scale != 1.0:
        img = cv2.resize(img, (int(w / scale), int(h / scale)))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    ws, hs = win_size
    for y in range(0, gray.shape[0] - hs + 1, stride):
        for x in range(0, gray.shape[1] - ws + 1, stride):
            feat        = hog.compute(clahe.apply(gray[y:y+hs, x:x+ws])).flatten()
            feat_pca    = pca.transform(feat.reshape(1, -1))
            feat_scaled = scaler.transform(feat_pca)
            score = clf.decision_function(feat_scaled)[0]
            if score > -0.5:
                detections.append((int(x*scale), int(y*scale),
                                   int((x+ws)*scale), int((y+hs)*scale), score))
    return detections


def detect(img, clf, hog, scaler, pca, scales=(1.0, 0.7, 0.5, 0.35), conf_thresh=-0.181):
    all_dets = [d for s in scales
                for d in sliding_window(img, clf, hog, scaler, pca, scale=s)
                if d[4] >= conf_thresh]
    filtered = []
    for x1, y1, x2, y2, conf in nms(all_dets):
        bw, bh = x2 - x1, y2 - y1
        if bh <= 0:
            continue
        aspect, area = bw / bh, bw * bh
        if not (0.5 < aspect < 2.5) or not (1200 < area < 150000):
            continue
        roi    = cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
        edges  = cv2.Canny(roi, 50, 150)
        edge_d = edges.sum() / (bw * bh * 255)
        if not (0.03 < edge_d < 0.7) or roi.var() < 150:
            continue
        filtered.append((x1, y1, x2, y2, conf))
    return filtered


print('HNM bootstrapping on train set (3 iterations)...')

X_current = X_pca.copy()
y_current = y.copy()

for iteration in range(3):
    print(f'
--- Iteration {iteration + 1} ---')
    hard_negs = []
    for img_path in df_train['img_path'].unique():
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_df = df_train[df_train['img_path'] == img_path]
        gt_boxes = [
            (int((r['cx']-r['w']/2)*W), int((r['cy']-r['h']/2)*H),
             int((r['cx']+r['w']/2)*W), int((r['cy']+r['h']/2)*H))
            for _, r in img_df.iterrows() if r['has_obj']
        ]
        for (x1, y1, x2, y2, _) in detect(img, clf, hog, scaler, pca, conf_thresh=-1.0):
            if all(iou((x1, y1, x2, y2), gt) <= 0.3 for gt in gt_boxes):
                crop = img[y1:y2, x1:x2]
                if crop.size > 0:
                    hard_negs.append(preprocess(crop))
    print(f'  Found {len(hard_negs)} hard negatives')
    if len(hard_negs) < 10:
        print('  Converged.')
        break

    X_hard_pca = pca.transform(np.array([hog.compute(s).flatten() for s in hard_negs]))
    X_current  = np.vstack([X_current, X_hard_pca])
    y_current  = np.hstack([y_current, np.zeros(len(hard_negs))])

    X_tr, X_v, y_tr, y_v = train_test_split(
        X_current, y_current, test_size=0.2, random_state=42, stratify=y_current
    )
    smote_     = SMOTE(random_state=42)
    X_r, y_r   = smote_.fit_resample(X_tr, y_tr)
    scaler_new = StandardScaler()
    X_r_s      = scaler_new.fit_transform(X_r)

    gs = GridSearchCV(
        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
        {'C': [0.01, 0.1, 1], 'gamma': ['scale', 'auto']},
        cv=StratifiedKFold(3, shuffle=True, random_state=42), scoring='f1', n_jobs=-1
    )
    gs.fit(X_r_s, y_r)
    print(f'  Best: {gs.best_params_}  CV F1: {gs.best_score_:.4f}')
    clf, scaler = gs.best_estimator_, scaler_new

pickle.dump(clf,    open(MODEL_DIR / 'svm_hog_v2.pkl', 'wb'))
pickle.dump(scaler, open(MODEL_DIR / 'scaler_v2.pkl',  'wb'))
pickle.dump(pca,    open(MODEL_DIR / 'pca.pkl',        'wb'))
print('
Final model v2 + scaler + PCA saved to models/')

## 13. Test Set Evaluation (Sliding Window vs GT)

In [ ]:
print('=' * 60)
print('SLIDING WINDOW EVALUATION - TEST SET')
print('=' * 60)

all_tp = all_fp = all_fn = 0

for img_path in splits['test']['img_path'].unique():
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_df = splits['test'][splits['test']['img_path'] == img_path]
    gt_boxes = [
        [int((r['cx']-r['w']/2)*W), int((r['cy']-r['h']/2)*H),
         int((r['cx']+r['w']/2)*W), int((r['cy']+r['h']/2)*H)]
        for _, r in img_df.iterrows() if r['has_obj']
    ]
    dets       = detect(img, clf, hog, scaler, pca)
    gt_matched = [False] * len(gt_boxes)

    for dx1, dy1, dx2, dy2, _ in dets:
        best_iou, best_gt = 0, -1
        for gi, gt in enumerate(gt_boxes):
            if not gt_matched[gi]:
                v = iou((dx1, dy1, dx2, dy2), gt)
                if v > best_iou:
                    best_iou, best_gt = v, gi
        if best_iou >= 0.4:
            gt_matched[best_gt] = True
            all_tp += 1
        else:
            all_fp += 1
    all_fn += sum(1 for m in gt_matched if not m)

prec = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
rec  = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

print(f'
TP={all_tp}  FP={all_fp}  FN={all_fn}')
print(f'Precision : {prec:.3f}')
print(f'Recall    : {rec:.3f}')
print(f'F1-score  : {f1:.3f}')

## 14. Detection Visualisation

In [ ]:
def draw_detections(img_path, detections, gt_boxes=None):
    img = cv2.imread(img_path)
    if img is None:
        return None
    if gt_boxes:
        for x1, y1, x2, y2 in gt_boxes:
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for x1, y1, x2, y2, conf in detections:
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        label = f'{conf:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x1, y1 - th - 4), (x1 + tw + 4, y1), (0, 0, 255), -1)
        cv2.putText(img, label, (x1 + 2, y1 - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


valid_imgs = list(splits['valid']['img_path'].unique())
samples    = random.sample(valid_imgs, min(6, len(valid_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, ipath in enumerate(samples):
    img_df   = splits['valid'][splits['valid']['img_path'] == ipath]
    gt_boxes = [
        (int((r['cx']-r['w']/2)*W), int((r['cy']-r['h']/2)*H),
         int((r['cx']+r['w']/2)*W), int((r['cy']+r['h']/2)*H))
        for _, r in img_df.iterrows() if r['has_obj']
    ]
    img  = cv2.imread(ipath)
    dets = detect(img, clf, hog, scaler, pca)
    axes[idx//3][idx%3].imshow(draw_detections(ipath, dets, gt_boxes))
    axes[idx//3][idx%3].set_title(f'{Path(ipath).name} - {len(dets)} detections')
    axes[idx//3][idx%3].axis('off')

plt.suptitle('Green = GT  |  Red = Detections', fontsize=14)
plt.tight_layout()
plt.show()

## 15. Video Detection

In [ ]:
def detect_video(video_path, output_path, clf, hog, scaler, pca, skip_frames=2):
    cap    = cv2.VideoCapture(video_path)
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % skip_frames == 0:
            for x1, y1, x2, y2, conf in detect(frame, clf, hog, scaler, pca):
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                label = f'{conf:.2f}'
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(frame, (x1, y1-th-4), (x1+tw+4, y1), (0, 0, 255), -1)
                cv2.putText(frame, label, (x1+2, y1-2),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f'  Processed {frame_count} frames...')

    cap.release(); out.release()
    print(f'Done! {frame_count} frames saved to {output_path}')

In [ ]:
import os

video_in  = 'dataset/Sample_Video_HighQuality.mp4'
video_out = 'output/detected.mp4'
os.makedirs('output', exist_ok=True)

print(f'Input : {video_in}')
print(f'Output: {video_out}')
detect_video(video_in, video_out, clf, hog, scaler, pca, skip_frames=2)